# LeetCode 1462 — Course Schedule IV bằng Python

Link đề: https://leetcode.com/problems/course-schedule-iv/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Graph Theory, Topological Sort

Skeleton LeetCode:

```python
class Solution:
    def checkIfPrerequisite(self, numCourses: int, prerequisites: list[list[int]], queries: list[list[int]]) -> list[bool]:
        ...
```

## 1. Mục tiêu học

1. Mô hình hóa "môn `a` là tiên quyết của môn `b`" thành **cạnh có hướng `a → b`** và "tiên quyết gián tiếp" thành **đường đi** trong đồ thị.
2. Trả lời truy vấn "`u` có tới được `v`?" bằng **DFS/BFS**, rồi nhận ra cần **tính trước bảng tới được** (bao đóng bắc cầu) khi có nhiều truy vấn.
3. Cài đặt **Floyd–Warshall** dạng boolean và hiểu vì sao vòng lặp trung gian `k` phải ở **ngoài cùng**.
4. Dùng **sắp xếp tô-pô (Kahn)** để lan truyền tập tiên quyết, biểu diễn tập bằng **bitmask số nguyên** cho nhanh.
5. Tránh lỗi: đảo chiều cạnh, chỉ xét cạnh trực tiếp, lan truyền thiếu tập của cha, kiểm tra bit sai.

## 2. Đề bài & ví dụ

Có `numCourses` môn học đánh số `0 .. numCourses−1`. Mảng `prerequisites[i] = [a_i, b_i]` nghĩa là **phải học `a_i` trước** khi học `b_i`. Quan hệ tiên quyết có tính **bắc cầu**: `a` trước `b`, `b` trước `c` ⇒ `a` là tiên quyết của `c`.

Cho `queries[j] = [u_j, v_j]`: trả lời `u_j` có phải là tiên quyết (trực tiếp hoặc gián tiếp) của `v_j` không. Trả về mảng boolean theo đúng thứ tự truy vấn.

**Ví dụ 1:** `numCourses = 2, prerequisites = [[1,0]], queries = [[0,1],[1,0]]` → `[False, True]`.

**Ví dụ 2:** `numCourses = 2, prerequisites = [], queries = [[1,0],[0,1]]` → `[False, False]`.

**Ví dụ 3:** `numCourses = 3, prerequisites = [[1,2],[1,0],[2,0]], queries = [[1,0],[1,2]]` → `[True, True]`.

**Ràng buộc:**
- `2 <= numCourses <= 100`
- `0 <= prerequisites.length <= n(n−1)/2`, các cặp phân biệt, `a_i != b_i`
- Đồ thị tiên quyết **không có chu trình** (DAG)
- `1 <= queries.length <= 10^4`, `u_j != v_j`

In [1]:
import random
import time
from collections import deque, defaultdict
from typing import List

examples = [
    (2, [[1, 0]], [[0, 1], [1, 0]], [False, True]),
    (2, [], [[1, 0], [0, 1]], [False, False]),
    (3, [[1, 2], [1, 0], [2, 0]], [[1, 0], [1, 2]], [True, True]),
]
for n, pre, qs, want in examples:
    print(f"n={n}, prerequisites={pre}, queries={qs} -> kỳ vọng {want}")

n=2, prerequisites=[[1, 0]], queries=[[0, 1], [1, 0]] -> kỳ vọng [False, True]
n=2, prerequisites=[], queries=[[1, 0], [0, 1]] -> kỳ vọng [False, False]
n=3, prerequisites=[[1, 2], [1, 0], [2, 0]], queries=[[1, 0], [1, 2]] -> kỳ vọng [True, True]


## 3. Kiến thức nền

### 3.1 Danh sách kề (adjacency list)

Lưu đồ thị có hướng bằng `adj[u] = [các v mà có cạnh u → v]`. Với cặp `[a, b]` ta thêm `b` vào `adj[a]` vì "học `a` xong mới học `b`" — đi theo chiều mũi tên là đi từ môn trước tới môn sau.

In [2]:
def build_adj(n, prerequisites):
    adj = [[] for _ in range(n)]
    for a, b in prerequisites:
        adj[a].append(b)          # a -> b
    return adj

print(build_adj(3, [[1, 2], [1, 0], [2, 0]]))   # 1 -> 2, 1 -> 0, 2 -> 0

[[], [2, 0], [0]]


### 3.2 BFS bằng `collections.deque`

`deque` cho phép `append` và `popleft` `O(1)`. BFS từ `s`: đưa `s` vào hàng đợi, lần lượt lấy ra và đẩy các đỉnh kề **chưa thăm**. Tập `visited` đảm bảo mỗi đỉnh vào hàng đợi tối đa 1 lần.

In [3]:
def bfs_reach(adj, s):
    seen = {s}
    q = deque([s])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if v not in seen:
                seen.add(v)
                q.append(v)
    return seen

adj = build_adj(4, [[0, 1], [1, 2], [3, 2]])
for s in range(4):
    print(f"từ {s} tới được: {sorted(bfs_reach(adj, s) - {s})}")

từ 0 tới được: [1, 2]
từ 1 tới được: [2]
từ 2 tới được: []
từ 3 tới được: [2]


### 3.3 DFS bằng stack (tránh đệ quy sâu)

DFS lặp dùng `list` làm stack (`append`/`pop` ở cuối). Thứ tự thăm khác BFS nhưng **tập đỉnh tới được là như nhau**.

In [4]:
def dfs_reach(adj, s):
    seen = {s}
    stack = [s]
    while stack:
        u = stack.pop()
        for v in adj[u]:
            if v not in seen:
                seen.add(v)
                stack.append(v)
    return seen

print([sorted(dfs_reach(adj, s)) == sorted(bfs_reach(adj, s)) for s in range(4)])

[True, True, True, True]


### 3.4 Ma trận boolean 2 chiều

`[[False] * n for _ in range(n)]` tạo `n` hàng **độc lập**. Không viết `[[False] * n] * n` — cách đó tạo `n` tham chiếu tới **cùng một** hàng.

In [5]:
good = [[False] * 3 for _ in range(3)]
bad = [[False] * 3] * 3
good[0][1] = True
bad[0][1] = True
print("good:", good)
print("bad :", bad, "<- cả 3 hàng đổi theo!")

good: [[False, True, False], [False, False, False], [False, False, False]]
bad : [[False, True, False], [False, True, False], [False, True, False]] <- cả 3 hàng đổi theo!


### 3.5 Bitmask số nguyên làm tập hợp

Python `int` không giới hạn bit, nên một số có thể biểu diễn tập con của `{0..99}`:
- thêm phần tử `x`: `mask |= 1 << x`
- hợp hai tập: `a | b`
- kiểm tra `x` thuộc tập: `(mask >> x) & 1`

Hợp hai tập 100 phần tử chỉ là **một phép OR** trên số ~100 bit — rất nhanh.

In [6]:
A = (1 << 0) | (1 << 5)          # {0, 5}
B = (1 << 5) | (1 << 99)         # {5, 99}
U = A | B
print(bin(A), bin(B)[:10] + "...")
print("U chứa:", [x for x in range(100) if (U >> x) & 1])
print("3 thuộc U?", (U >> 3) & 1, "| 99 thuộc U?", (U >> 99) & 1)

0b100001 0b10000000...
U chứa: [0, 5, 99]
3 thuộc U? 0 | 99 thuộc U? 1


### 3.6 Bậc vào (in-degree) và sắp xếp tô-pô Kahn

`indeg[v]` = số cạnh đi **vào** `v`. Kahn: đưa mọi đỉnh `indeg = 0` vào hàng đợi; lấy `u` ra, "xóa" các cạnh `u → v` (giảm `indeg[v]`), đỉnh nào về 0 thì vào hàng đợi. Thứ tự lấy ra là **thứ tự tô-pô**: mọi cạnh `u → v` có `u` đứng trước `v`.

In [7]:
def kahn_order(n, prerequisites):
    adj = build_adj(n, prerequisites)
    indeg = [0] * n
    for a, b in prerequisites:
        indeg[b] += 1
    q = deque(i for i in range(n) if indeg[i] == 0)
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return order

print(kahn_order(3, [[1, 2], [1, 0], [2, 0]]))   # 1 phải trước 2, 2 trước 0

[1, 2, 0]


## 4. Ý tưởng nền tảng / trực giác

Vẽ ví dụ 3: cạnh `1 → 2`, `1 → 0`, `2 → 0`.

```
  1 ──► 2
   \    │
    ▼   ▼
      0
```

- Truy vấn `[1, 0]`: có cạnh trực tiếp `1 → 0` ⇒ `True`.
- Truy vấn `[1, 2]`: cạnh `1 → 2` ⇒ `True`.
- Nếu hỏi `[0, 1]`: từ `0` không có cạnh đi ra ⇒ `False`.

Tổng quát: **`u` là tiên quyết của `v` ⇔ có đường đi có hướng từ `u` tới `v`**. Bài toán thành "nhiều truy vấn tới được (reachability)". Tính tay bảng `reach[u][v]` cho ví dụ 3:

| từ \ tới | 0 | 1 | 2 |
|---|---|---|---|
| 0 | – | ✘ | ✘ |
| 1 | ✔ | – | ✔ |
| 2 | ✔ | ✘ | – |

Có tới `10^4` truy vấn nhưng chỉ `100` đỉnh ⇒ chỉ có tối đa `100 × 99` cặp khác nhau. Tính sẵn cả bảng rồi tra là hợp lý.

In [8]:
n, pre, qs, want = examples[2]
adj = build_adj(n, pre)
table = [[v in bfs_reach(adj, u) and u != v for v in range(n)] for u in range(n)]
for u in range(n):
    print(u, ["✔" if x else "✘" for x in table[u]])

0 ['✘', '✘', '✘']
1 ['✔', '✘', '✔']
2 ['✔', '✘', '✘']


## 5. Cách 1 — Brute Force / baseline: BFS riêng cho mỗi truy vấn

**Ý tưởng:** với từng truy vấn `[u, v]`, chạy BFS từ `u`, dừng ngay khi gặp `v`.

Đơn giản và đúng, dùng làm chuẩn đối chiếu.

In [9]:
def check_brute(numCourses, prerequisites, queries):
    adj = build_adj(numCourses, prerequisites)
    ans = []
    for u, v in queries:
        seen = {u}
        q = deque([u])
        found = False
        while q and not found:
            x = q.popleft()
            for y in adj[x]:
                if y == v:
                    found = True
                    break
                if y not in seen:
                    seen.add(y)
                    q.append(y)
        ans.append(found)
    return ans

for n, pre, qs, want in examples:
    got = check_brute(n, pre, qs)
    print(got, want, "✔" if got == want else "✘")

[False, True] [False, True] ✔
[False, False] [False, False] ✔
[True, True] [True, True] ✔


**Độ phức tạp:** mỗi truy vấn tốn `O(n + E)`, với `E ≤ n(n−1)/2 ≈ 4950`. Tổng `O(q · (n + E))`, xấu nhất ≈ `10^4 × 5·10^3 = 5·10^7` thao tác. Truy vấn trả `True` thường dừng sớm, nhưng truy vấn trả `False` buộc BFS duyệt hết vùng tới được — trong Python đó là phần tốn thời gian (đo ở cell dưới).

**Nút thắt:** nhiều truy vấn có cùng `u` ⇒ BFS từ `u` bị chạy lại nhiều lần. Kết quả BFS từ `u` (tập đỉnh tới được) không đổi giữa các truy vấn ⇒ nên **tính một lần, dùng nhiều lần**.

In [10]:
def random_dag(n, p, rng):
    perm = list(range(n))
    rng.shuffle(perm)                     # thứ tự tô-pô ẩn
    return [[perm[i], perm[j]] for i in range(n) for j in range(i + 1, n) if rng.random() < p]

rng = random.Random(1462)
n_big = 100
pre_big = random_dag(n_big, 0.5, rng)
q_big = []
while len(q_big) < 2000:
    u, v = rng.randrange(n_big), rng.randrange(n_big)
    if u != v:
        q_big.append([u, v])
t = time.perf_counter()
check_brute(n_big, pre_big, q_big)
print(f"E={len(pre_big)}, 2000 truy vấn ngẫu nhiên: {time.perf_counter() - t:.3f}s (nhiều truy vấn True dừng sớm)")
adj_big = build_adj(n_big, pre_big)
q_false = [[u, v] for u, v in q_big if v not in bfs_reach(adj_big, u)]
t = time.perf_counter()
check_brute(n_big, pre_big, q_false)
dt = time.perf_counter() - t
print(f"{len(q_false)} truy vấn False (BFS phải chạy hết): {dt:.3f}s -> ước tính 10^4 truy vấn: {dt / len(q_false) * 10**4:.2f}s")

E=2513, 2000 truy vấn ngẫu nhiên: 0.021s (nhiều truy vấn True dừng sớm)
1038 truy vấn False (BFS phải chạy hết): 0.016s -> ước tính 10^4 truy vấn: 0.16s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** số đỉnh nhỏ (`n ≤ 100`) còn số truy vấn lớn ⇒ **tính trước bao đóng bắc cầu** `reach[u][v]` (có đường từ `u` tới `v`), sau đó mỗi truy vấn tra `O(1)`.

Ba cách tính bao đóng:
1. **BFS từ mỗi đỉnh:** `n` lần BFS, `O(n · (n + E))`.
2. **Floyd–Warshall:** quy tắc "**nếu** `u` tới được `k` **và** `k` tới được `v` **thì** `u` tới được `v`", lặp `k` ở ngoài cùng. `O(n³) = 10^6`.
3. **Tô-pô + tập tiên quyết:** gọi `pre[v]` = tập mọi tiên quyết của `v`. **Nếu** có cạnh `u → v` **thì** `pre[v] ⊇ pre[u] ∪ {u}`. Duyệt theo thứ tự tô-pô thì khi xử lý `v`, mọi `pre[u]` của cha đã hoàn chỉnh. Dùng bitmask ⇒ mỗi cạnh một phép OR.

**Invariant (Kahn):** khi đỉnh `u` được lấy khỏi hàng đợi, `pre[u]` đã chứa **đủ** mọi tiên quyết của `u` (vì mọi cha của `u` đã được xử lý trước — đó là lý do `indeg[u]` về 0).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — BFS từ mọi đỉnh, lưu bảng `reach`

**Từng đoạn:**
1. Dựng `adj`.
2. Với mỗi `s`, BFS và đánh dấu `reach[s][v] = True` cho mọi `v` thăm được (gợi ý chính thức của đề).
3. Trả lời `reach[u][v]` cho từng truy vấn.

In [11]:
def check_bfs_all(numCourses, prerequisites, queries):
    adj = build_adj(numCourses, prerequisites)
    reach = [[False] * numCourses for _ in range(numCourses)]
    for s in range(numCourses):
        q = deque([s])
        while q:
            x = q.popleft()
            for y in adj[x]:
                if not reach[s][y]:
                    reach[s][y] = True
                    q.append(y)
    return [reach[u][v] for u, v in queries]

for n, pre, qs, want in examples:
    print(check_bfs_all(n, pre, qs), want)

[False, True] [False, True]
[False, False] [False, False]
[True, True] [True, True]


Ở đây `reach[s]` đóng vai trò `visited` cho BFS từ `s` (đồ thị là DAG nên `s` không bao giờ tự tới lại chính nó). Code nộp LeetCode:

In [12]:
class Solution:
    def checkIfPrerequisite(self, numCourses: int, prerequisites: List[List[int]],
                            queries: List[List[int]]) -> List[bool]:
        adj = [[] for _ in range(numCourses)]
        for a, b in prerequisites:
            adj[a].append(b)
        reach = [[False] * numCourses for _ in range(numCourses)]
        for s in range(numCourses):
            q = deque([s])
            while q:
                x = q.popleft()
                for y in adj[x]:
                    if not reach[s][y]:
                        reach[s][y] = True
                        q.append(y)
        return [reach[u][v] for u, v in queries]

print([Solution().checkIfPrerequisite(n, p, q) for n, p, q, _ in examples])

[[False, True], [False, False], [True, True]]


**Độ phức tạp:** `O(n · (n + E) + q)` thời gian, `O(n² + E)` bộ nhớ. Với `n = 100`: tối đa ~`5·10^5` bước BFS.

### 7.2 Cách 3 — Floyd–Warshall bao đóng bắc cầu

**Từng đoạn:**
1. Khởi tạo `reach[a][b] = True` cho mọi cạnh.
2. Với từng đỉnh trung gian `k` (vòng **ngoài cùng**), với mọi `i`, `j`: nếu `reach[i][k] and reach[k][j]` thì `reach[i][j] = True`.
3. Sau vòng `k`, `reach[i][j]` đúng cho mọi đường đi chỉ dùng đỉnh trung gian thuộc `{0..k}`.

Tối ưu nhỏ: nếu `reach[i][k]` sai thì bỏ qua cả hàng `j`.

In [13]:
def check_floyd(numCourses, prerequisites, queries):
    n = numCourses
    reach = [[False] * n for _ in range(n)]
    for a, b in prerequisites:
        reach[a][b] = True
    for k in range(n):
        rk = reach[k]
        for i in range(n):
            if reach[i][k]:
                ri = reach[i]
                for j in range(n):
                    if rk[j]:
                        ri[j] = True
    return [reach[u][v] for u, v in queries]

for n, pre, qs, want in examples:
    print(check_floyd(n, pre, qs), want)

[False, True] [False, True]
[False, False] [False, False]
[True, True] [True, True]


Code nộp LeetCode (Cách 3):

In [14]:
class Solution:
    def checkIfPrerequisite(self, numCourses: int, prerequisites: List[List[int]],
                            queries: List[List[int]]) -> List[bool]:
        n = numCourses
        reach = [[False] * n for _ in range(n)]
        for a, b in prerequisites:
            reach[a][b] = True
        for k in range(n):
            for i in range(n):
                if reach[i][k]:
                    for j in range(n):
                        if reach[k][j]:
                            reach[i][j] = True
        return [reach[u][v] for u, v in queries]

print([Solution().checkIfPrerequisite(n, p, q) for n, p, q, _ in examples])

[[False, True], [False, False], [True, True]]


**Độ phức tạp:** `O(n³ + q)` thời gian (`10^6` với `n = 100`), `O(n²)` bộ nhớ. Code ngắn, không cần danh sách kề.

### 7.3 Cách 4 — Kahn + bitmask tập tiên quyết

**Từng đoạn:**
1. Dựng `adj`, `indeg`; `pre = [0] * n` (bitmask rỗng).
2. Hàng đợi chứa các đỉnh `indeg = 0` (môn không cần tiên quyết).
3. Lấy `u`; với mỗi cạnh `u → v`: `pre[v] |= pre[u] | (1 << u)` — `v` thừa hưởng mọi tiên quyết của `u` và chính `u`. Giảm `indeg[v]`, về 0 thì đẩy vào hàng đợi.
4. Truy vấn `[u, v]`: `(pre[v] >> u) & 1`.

In [15]:
def check_kahn_bitmask(numCourses, prerequisites, queries):
    n = numCourses
    adj = [[] for _ in range(n)]
    indeg = [0] * n
    for a, b in prerequisites:
        adj[a].append(b)
        indeg[b] += 1
    pre = [0] * n
    q = deque(i for i in range(n) if indeg[i] == 0)
    while q:
        u = q.popleft()
        for v in adj[u]:
            pre[v] |= pre[u] | (1 << u)
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return [bool((pre[v] >> u) & 1) for u, v in queries]

for n, pre_, qs, want in examples:
    print(check_kahn_bitmask(n, pre_, qs), want)

[False, True] [False, True]
[False, False] [False, False]
[True, True] [True, True]


Code nộp LeetCode (Cách 4):

In [16]:
class Solution:
    def checkIfPrerequisite(self, numCourses: int, prerequisites: List[List[int]],
                            queries: List[List[int]]) -> List[bool]:
        adj = [[] for _ in range(numCourses)]
        indeg = [0] * numCourses
        for a, b in prerequisites:
            adj[a].append(b)
            indeg[b] += 1
        pre = [0] * numCourses
        q = deque(i for i in range(numCourses) if indeg[i] == 0)
        while q:
            u = q.popleft()
            for v in adj[u]:
                pre[v] |= pre[u] | (1 << u)
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        return [bool((pre[v] >> u) & 1) for u, v in queries]

print([Solution().checkIfPrerequisite(n, p, q) for n, p, q, _ in examples])

[[False, True], [False, False], [True, True]]


**Độ phức tạp:** `O(n + E)` phép OR trên số `n` bit (mỗi phép ~`O(n/64)` máy) + `O(q)` truy vấn. Bộ nhớ `O(n + E)` (mỗi bitmask ~`n` bit). Đây là cách nhanh nhất trong thực tế.

So sánh thời gian trên đồ thị dày `n = 100` và `10^4` truy vấn:

In [17]:
q_full = q_big * 5
for name, f in [("BFS mỗi đỉnh", check_bfs_all), ("Floyd–Warshall", check_floyd),
                ("Kahn + bitmask", check_kahn_bitmask)]:
    t = time.perf_counter()
    r = f(n_big, pre_big, q_full)
    print(f"{name:15s}: {time.perf_counter() - t:.3f}s, số True = {sum(r)}")

BFS mỗi đỉnh   : 0.004s, số True = 4810
Floyd–Warshall : 0.013s, số True = 4810
Kahn + bitmask : 0.002s, số True = 4810


## 8. TRACE từng bước

Trace Cách 4 với `n = 4`, `prerequisites = [[0,1],[1,2],[3,2],[0,3]]`, tức các cạnh `0 → 1`, `1 → 2`, `3 → 2`, `0 → 3` (`adj[0] = [1, 3]`):

| bước | lấy u | cạnh xử lý | pre sau bước (dạng tập) | hàng đợi |
|---|---|---|---|---|
| khởi tạo | – | – | pre = [∅, ∅, ∅, ∅] | [0] |
| 1 | 0 | 0→1, 0→3 | pre[1]={0}, pre[3]={0} | [1, 3] |
| 2 | 1 | 1→2 | pre[2]={0,1} (indeg[2]: 2→1) | [3] |
| 3 | 3 | 3→2 | pre[2]={0,1,3} (indeg[2]→0) | [2] |
| 4 | 2 | – | – | [] |

Hàm `trace_kahn` in các biến này.

In [18]:
def trace_kahn(n, prerequisites):
    adj = [[] for _ in range(n)]
    indeg = [0] * n
    for a, b in prerequisites:
        adj[a].append(b)
        indeg[b] += 1
    pre = [0] * n
    show = lambda m: "{" + ",".join(str(x) for x in range(n) if (m >> x) & 1) + "}"
    q = deque(i for i in range(n) if indeg[i] == 0)
    print("khởi tạo: indeg =", indeg, "| hàng đợi =", list(q))
    step = 0
    while q:
        u = q.popleft()
        step += 1
        print(f"bước {step}: lấy u={u}, pre[{u}]={show(pre[u])}")
        for v in adj[u]:
            pre[v] |= pre[u] | (1 << u)
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
            print(f"   cạnh {u}->{v}: pre[{v}]={show(pre[v])}, indeg[{v}]={indeg[v]}, hàng đợi={list(q)}")
    print("kết quả pre:", [show(m) for m in pre])
    return pre

_ = trace_kahn(4, [[0, 1], [1, 2], [3, 2], [0, 3]])

khởi tạo: indeg = [0, 1, 2, 1] | hàng đợi = [0]
bước 1: lấy u=0, pre[0]={}
   cạnh 0->1: pre[1]={0}, indeg[1]=0, hàng đợi=[1]
   cạnh 0->3: pre[3]={0}, indeg[3]=0, hàng đợi=[1, 3]
bước 2: lấy u=1, pre[1]={0}
   cạnh 1->2: pre[2]={0,1}, indeg[2]=1, hàng đợi=[3]
bước 3: lấy u=3, pre[3]={0}
   cạnh 3->2: pre[2]={0,1,3}, indeg[2]=0, hàng đợi=[2]
bước 4: lấy u=2, pre[2]={0,1,3}
kết quả pre: ['{}', '{0}', '{0,1,3}', '{0}']


Trace Floyd–Warshall trên ví dụ 3 (in những ô mới bật ở mỗi `k`):

In [19]:
def trace_floyd(n, prerequisites):
    reach = [[False] * n for _ in range(n)]
    for a, b in prerequisites:
        reach[a][b] = True
    print("ban đầu:", [(i, j) for i in range(n) for j in range(n) if reach[i][j]])
    for k in range(n):
        new = []
        for i in range(n):
            for j in range(n):
                if reach[i][k] and reach[k][j] and not reach[i][j]:
                    reach[i][j] = True
                    new.append((i, j))
        print(f"k={k}: bật thêm {new}")
    return reach

_ = trace_floyd(3, [[1, 2], [1, 0], [2, 0]])
print()
_ = trace_floyd(4, [[1, 3], [3, 2], [2, 0]])   # chuỗi 1 -> 3 -> 2 -> 0

ban đầu: [(1, 0), (1, 2), (2, 0)]
k=0: bật thêm []
k=1: bật thêm []
k=2: bật thêm []

ban đầu: [(1, 3), (2, 0), (3, 2)]
k=0: bật thêm []
k=1: bật thêm []
k=2: bật thêm [(3, 0)]
k=3: bật thêm [(1, 0), (1, 2)]


## 9. Code nộp LeetCode

Bản khuyên dùng (Kahn + bitmask), đúng chữ ký `checkIfPrerequisite(self, numCourses, prerequisites, queries)`:

In [20]:
class Solution:
    def checkIfPrerequisite(self, numCourses: int, prerequisites: List[List[int]],
                            queries: List[List[int]]) -> List[bool]:
        adj = [[] for _ in range(numCourses)]
        indeg = [0] * numCourses
        for a, b in prerequisites:
            adj[a].append(b)
            indeg[b] += 1
        pre = [0] * numCourses               # pre[v]: bitmask các tiên quyết của v
        q = deque(i for i in range(numCourses) if indeg[i] == 0)
        while q:
            u = q.popleft()
            for v in adj[u]:
                pre[v] |= pre[u] | (1 << u)
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        return [bool((pre[v] >> u) & 1) for u, v in queries]

for n, p, q, want in examples:
    assert Solution().checkIfPrerequisite(n, p, q) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

- **Mô hình:** theo định nghĩa bắc cầu, `u` là tiên quyết của `v` ⇔ tồn tại dãy `u = x0 → x1 → … → xt = v` các cạnh tiên quyết ⇔ có đường đi có hướng `u ⇝ v`.
- **BFS từ mỗi đỉnh:** BFS thăm đúng tập đỉnh tới được từ nguồn ⇒ `reach[s][v]` chính xác.
- **Floyd–Warshall:** Bất biến sau vòng `k`: `reach[i][j]` đúng ⇔ có đường `i ⇝ j` mà mọi đỉnh trung gian thuộc `{0..k}`. Một đường như vậy hoặc không qua `k` (đã đúng từ trước), hoặc qua `k` đúng một lần (DAG/đường đơn) ⇒ tách thành `i ⇝ k` và `k ⇝ j` chỉ dùng trung gian `{0..k−1}`, đã đúng ở vòng trước. Vì vậy `k` phải ở **ngoài cùng**.
- **Kahn + bitmask:** Quy nạp theo thứ tự tô-pô. Tập tiên quyết của `v` = hợp trên mọi cha `u` của `pre[u] ∪ {u}`. Một đỉnh chỉ vào hàng đợi khi mọi cha đã được lấy ra và đã OR vào `pre[v]` ⇒ khi `v` được dùng để lan tiếp, `pre[v]` đã đủ. Đồ thị là DAG nên mọi đỉnh đều được lấy ra.

## 11. Phân tích Time / Space

Gọi `n = numCourses ≤ 100`, `E = len(prerequisites) ≤ 4950`, `q = len(queries) ≤ 10^4`.

| Cách | Time | Space |
|---|---|---|
| 1. BFS mỗi truy vấn | `O(q · (n + E))` ≈ `5·10^7` | `O(n + E)` |
| 2. BFS mỗi đỉnh | `O(n · (n + E) + q)` ≈ `5·10^5` | `O(n² + E)` |
| 3. Floyd–Warshall | `O(n³ + q)` = `10^6` | `O(n²)` |
| 4. Kahn + bitmask | `O((n + E) · n/w + q)`, `w` = 64 bit | `O(n² / w + E)` |

Không có khác biệt average/worst đáng kể; trường hợp xấu là đồ thị dày (`E` cỡ `n²/2`).

## 12. Bộ test

Đối chiếu 4 cách trên các ví dụ, edge case (không có cạnh, chuỗi dài, đồ thị đầy đủ theo thứ tự tô-pô), và DAG ngẫu nhiên với seed cố định.

In [21]:
ALL = [check_brute, check_bfs_all, check_floyd, check_kahn_bitmask,
       lambda n, p, q: Solution().checkIfPrerequisite(n, p, q)]

def check_all(n, pre, qs, want=None):
    res = [f(n, pre, qs) for f in ALL]
    if want is None:
        want = res[0]
    for r in res:
        assert r == want, (n, pre, qs, res)
    return want

for n, p, q, want in examples:
    check_all(n, p, q, want)

# chuỗi 0 -> 1 -> ... -> 99
chain = [[i, i + 1] for i in range(99)]
assert check_all(100, chain, [[0, 99], [99, 0], [50, 51], [51, 50]]) == [True, False, True, False]
# đồ thị đầy đủ theo thứ tự tô-pô 0 < 1 < ... < 9
full = [[i, j] for i in range(10) for j in range(i + 1, 10)]
assert check_all(10, full, [[i, j] for i in range(10) for j in range(10) if i != j]) == \
       [i < j for i in range(10) for j in range(10) if i != j]
# không cạnh
assert check_all(5, [], [[0, 1], [4, 3]]) == [False, False]
# chuỗi có nhãn xáo trộn 1 -> 3 -> 2 -> 0
assert check_all(4, [[1, 3], [3, 2], [2, 0]], [[1, 0], [0, 1], [3, 0]]) == [True, False, True]
print("Edge case: OK")

Edge case: OK


In [22]:
rng = random.Random(42)
total_true = 0
for _ in range(400):
    n = rng.randint(2, 12)
    pre = random_dag(n, rng.choice([0.1, 0.3, 0.6]), rng)
    qs = []
    for _ in range(rng.randint(1, 20)):
        u, v = rng.sample(range(n), 2)
        qs.append([u, v])
    total_true += sum(check_all(n, pre, qs))
print("Random 400 DAG: OK, tổng số True =", total_true)

Random 400 DAG: OK, tổng số True = 864


In [23]:
# Ràng buộc lớn: n = 100, đồ thị gần đầy đủ, 10^4 truy vấn
rng = random.Random(7)
pre_max = random_dag(100, 0.95, rng)
qs_max = [rng.sample(range(100), 2) for _ in range(10**4)]
t = time.perf_counter()
r = Solution().checkIfPrerequisite(100, pre_max, qs_max)
print(f"E={len(pre_max)}, q=10^4: {time.perf_counter() - t:.3f}s, số True = {sum(r)}")
assert r == check_floyd(100, pre_max, qs_max)

E=4699, q=10^4: 0.003s, số True = 5071


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đảo chiều cạnh

`[a, b]` nghĩa là `a` trước `b`, tức cạnh `a → b`. Dựng `adj[b].append(a)` rồi vẫn hỏi "`u` tới `v`?" sẽ cho kết quả ngược.

In [24]:
def wrong_reversed(numCourses, prerequisites, queries):
    adj = [[] for _ in range(numCourses)]
    for a, b in prerequisites:
        adj[b].append(a)                     # SAI chiều
    return [v in bfs_reach(adj, u) for u, v in queries]

n, p, q, want = examples[0]
print("SAI :", wrong_reversed(n, p, q), "| đúng:", want)

SAI : [True, False] | đúng: [False, True]


**Sửa:** `adj[a].append(b)`. (Hoặc nếu đã dựng ngược thì hỏi "`v` tới `u`?" — nhưng phải nhất quán.)

### Lỗi 2 — Chỉ xét cạnh trực tiếp

Quên tính bắc cầu: chỉ kiểm tra `[u, v]` có trong `prerequisites`.

In [25]:
def wrong_direct_only(numCourses, prerequisites, queries):
    edges = {tuple(e) for e in prerequisites}
    return [(u, v) in edges for u, v in queries]   # SAI: bỏ qua đường gián tiếp

n, p, q = 3, [[0, 1], [1, 2]], [[0, 2]]
print("SAI :", wrong_direct_only(n, p, q), "| đúng:", check_floyd(n, p, q))

SAI : [False] | đúng: [True]


**Sửa:** tính bao đóng bắc cầu (BFS/Floyd/Kahn).

### Lỗi 3 — Floyd–Warshall đặt vòng `k` ở trong cùng

Viết `for i: for j: for k:` trông "đối xứng" nhưng sai: khi xét `(i, j)`, các ô `reach[k][j]` của hàng `k > i` **chưa được cập nhật**.

In [26]:
def wrong_floyd_order(numCourses, prerequisites, queries):
    n = numCourses
    reach = [[False] * n for _ in range(n)]
    for a, b in prerequisites:
        reach[a][b] = True
    for i in range(n):
        for j in range(n):
            for k in range(n):                # SAI: k phải ở ngoài cùng
                if reach[i][k] and reach[k][j]:
                    reach[i][j] = True
    return [reach[u][v] for u, v in queries]

n, p, q = 4, [[1, 3], [3, 2], [2, 0]], [[1, 0]]
print("SAI :", wrong_floyd_order(n, p, q), "| đúng:", check_floyd(n, p, q))
print("Khi xét (1,0), hàng 3 chưa biết 3 tới 0 (cần qua 2) nên bỏ sót.")

SAI : [False] | đúng: [True]
Khi xét (1,0), hàng 3 chưa biết 3 tới 0 (cần qua 2) nên bỏ sót.


**Sửa:** `for k: for i: for j:`.

### Lỗi 4 — Lan truyền thiếu tập của cha trong Kahn

Chỉ thêm `u` vào `pre[v]` (`pre[v] |= 1 << u`) mà quên `pre[u]` ⇒ chỉ biết cha trực tiếp.

In [27]:
def wrong_kahn_no_inherit(numCourses, prerequisites, queries):
    n = numCourses
    adj = [[] for _ in range(n)]
    indeg = [0] * n
    for a, b in prerequisites:
        adj[a].append(b); indeg[b] += 1
    pre = [0] * n
    q = deque(i for i in range(n) if indeg[i] == 0)
    while q:
        u = q.popleft()
        for v in adj[u]:
            pre[v] |= 1 << u                  # SAI: thiếu pre[u]
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return [bool((pre[v] >> u) & 1) for u, v in queries]

n, p, q = 3, [[0, 1], [1, 2]], [[0, 2]]
print("SAI :", wrong_kahn_no_inherit(n, p, q), "| đúng:", check_kahn_bitmask(n, p, q))

SAI : [False] | đúng: [True]


**Sửa:** `pre[v] |= pre[u] | (1 << u)`.

### Lỗi 5 — Kiểm tra bit sai: `pre[v] & u`

`pre[v] & u` lấy AND với **số** `u`, không phải với bit thứ `u`. Với `u = 0` luôn ra 0.

In [28]:
def wrong_bit_test(numCourses, prerequisites, queries):
    n = numCourses
    adj = [[] for _ in range(n)]
    indeg = [0] * n
    for a, b in prerequisites:
        adj[a].append(b); indeg[b] += 1
    pre = [0] * n
    q = deque(i for i in range(n) if indeg[i] == 0)
    while q:
        u = q.popleft()
        for v in adj[u]:
            pre[v] |= pre[u] | (1 << u)
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return [bool(pre[v] & u) for u, v in queries]   # SAI

n, p, q, want = examples[0]
print("ví dụ 1 — SAI :", wrong_bit_test(n, p, q), "| đúng:", want)
n, p, q = 3, [[0, 2]], [[1, 2]]
print("phản ví dụ 2 — SAI :", wrong_bit_test(n, p, q), "| đúng:", check_floyd(n, p, q),
      "(pre[2] = 0b1, 0b1 & 1 = 1 dù 1 không phải tiên quyết)")

ví dụ 1 — SAI : [False, False] | đúng: [False, True]
phản ví dụ 2 — SAI : [True] | đúng: [False] (pre[2] = 0b1, 0b1 & 1 = 1 dù 1 không phải tiên quyết)


**Sửa:** `(pre[v] >> u) & 1` hoặc `pre[v] & (1 << u)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tiên quyết gián tiếp", "có đường đi từ … tới …", nhiều truy vấn ⇒ **bao đóng bắc cầu**.
- `n ≤ 100` (hoặc ≤ 400) ⇒ Floyd–Warshall `O(n³)` thoải mái; `n` lớn hơn ⇒ BFS từ mỗi đỉnh hoặc tô-pô + bitset.
- Câu "đồ thị không có chu trình" ⇒ nghĩ ngay tới **thứ tự tô-pô** và DP trên DAG.
- Tập con của `{0..n−1}` với `n` nhỏ ⇒ **bitmask `int`** của Python: hợp = `|`, kiểm tra = `>> x & 1`.
- Luôn tự hỏi: cạnh `[a, b]` là `a → b` hay `b → a`? Viết ra 1 dòng chú thích trước khi code.

## 15. Biến thể của bài

1. **207 / 210 — Course Schedule I / II:** có học hết được không (phát hiện chu trình), in một thứ tự học.
2. **Đếm số tiên quyết của mỗi môn:** `pre[v].bit_count()` sau Cách 4 (bài tập 4).
3. **2192 — All Ancestors of a Node in a DAG:** liệt kê mọi tổ tiên của từng đỉnh, sắp tăng — chính là `pre[v]` dưới dạng danh sách.
4. **Truy vấn động** (thêm cạnh xen giữa truy vấn): cập nhật bao đóng tăng dần: thêm `a → b` thì mọi `x` tới `a` sẽ tới mọi `y` mà `b` tới.
5. **Đồ thị có chu trình:** nén thành phần liên thông mạnh (Tarjan/Kosaraju) rồi làm trên DAG nén.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS mỗi truy vấn | Tìm đường cho từng `[u, v]` | `O(q(n+E))` | `O(n+E)` | Ít truy vấn, đối chiếu |
| 2. BFS mỗi đỉnh | Tính trước `reach[s][*]` | `O(n(n+E)+q)` | `O(n²)` | Gợi ý chuẩn, dễ giải thích |
| 3. Floyd–Warshall | `i⇝k ∧ k⇝j ⇒ i⇝j`, `k` ngoài | `O(n³+q)` | `O(n²)` | `n` nhỏ, code ngắn nhất |
| 4. Kahn + bitmask | Lan tập tiên quyết theo tô-pô | `O((n+E)·n/w+q)` | `O(n²/w+E)` | Nhanh nhất, tận dụng DAG |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `numCourses = 4, prerequisites = [[2,3],[3,1],[0,1]]`, `queries = [[2,1],[1,2],[0,3]]`. Kết quả?

**Bài 2 (tính tay):** Với dữ liệu bài 1, viết bảng `pre[v]` (dạng tập) sau khi chạy Cách 4.

**Bài 3 (điền TODO):** Viết `all_ancestors(n, edges)` trả về danh sách, phần tử thứ `v` là **danh sách tăng dần** các tổ tiên của `v` (giống LeetCode 2192).

**Bài 4 (tự viết lại):** Viết `count_prereqs(n, prerequisites)` trả về mảng `cnt[v]` = số môn tiên quyết (trực tiếp + gián tiếp) của `v`, dùng Floyd–Warshall.

**Bài 5 (mini challenge):** Hỗ trợ **thêm cạnh động**: viết lớp `Closure(n)` với `add(a, b)` (đảm bảo vẫn là DAG) và `query(u, v)`. Mỗi `add` phải cập nhật bao đóng trong `O(n²)` hoặc tốt hơn bằng bitmask.

In [29]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def all_ancestors(n: int, edges: List[List[int]]) -> List[List[int]]:
    # TODO: Kahn + bitmask, rồi đổi bitmask thành danh sách tăng dần
    raise NotImplementedError

# Bài 4 — TODO
def count_prereqs(n: int, prerequisites: List[List[int]]) -> List[int]:
    # TODO: Floyd–Warshall rồi đếm theo cột
    raise NotImplementedError

# Bài 5 — TODO
class ClosureTodo:
    def __init__(self, n: int):
        # TODO: anc[v] (bitmask tổ tiên), des[u] (bitmask hậu duệ)
        raise NotImplementedError
    def add(self, a: int, b: int) -> None:
        raise NotImplementedError
    def query(self, u: int, v: int) -> bool:
        raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Vẽ cạnh `2→3→1`, `0→1`. (2) `2` tới `1` qua `3`. (3) `0` chỉ tới `1`.

**Bài 2.** (1) Thứ tự tô-pô: `0, 2, 3, 1`. (2) `pre[3] = {2}`. (3) `pre[1] = {0, 2, 3}`.

**Bài 3.** (1) Dùng lại `check_kahn_bitmask` nhưng trả về `pre`. (2) Duyệt `x` từ `0` tới `n−1`. (3) `[x for x in range(n) if pre[v] >> x & 1]`.

**Bài 4.** (1) `reach` sau Floyd. (2) `cnt[v] = Σ_u reach[u][v]`. (3) Dùng `sum(reach[u][v] for u in range(n))`.

**Bài 5.** (1) Giữ `anc[v]` = bitmask tổ tiên của `v` (kể cả chính nó), `des[u]` = bitmask hậu duệ (kể cả chính nó). (2) Thêm `a → b`: mọi `y ∈ des[b]` có thêm tổ tiên `anc[a]`; mọi `x ∈ anc[a]` có thêm hậu duệ `des[b]`. (3) `query(u, v)`: `u != v and anc[v] >> u & 1`.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1 + Bài 2
n1, p1, q1 = 4, [[2, 3], [3, 1], [0, 1]], [[2, 1], [1, 2], [0, 3]]
ans1 = check_kahn_bitmask(n1, p1, q1)
print("Bài 1:", ans1)
assert ans1 == [True, False, False]

def pre_sets(n, prerequisites):
    adj = [[] for _ in range(n)]
    indeg = [0] * n
    for a, b in prerequisites:
        adj[a].append(b); indeg[b] += 1
    pre = [0] * n
    q = deque(i for i in range(n) if indeg[i] == 0)
    while q:
        u = q.popleft()
        for v in adj[u]:
            pre[v] |= pre[u] | (1 << u)
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return pre

sets1 = [{x for x in range(n1) if m >> x & 1} for m in pre_sets(n1, p1)]
print("Bài 2:", sets1)
assert sets1 == [set(), {0, 2, 3}, set(), {2}]

Bài 1: [True, False, False]
Bài 2: [set(), {0, 2, 3}, set(), {2}]


In [31]:
def all_ancestors_ans(n, edges):
    pre = pre_sets(n, edges)
    return [[x for x in range(n) if pre[v] >> x & 1] for v in range(n)]

def count_prereqs_ans(n, prerequisites):
    reach = [[False] * n for _ in range(n)]
    for a, b in prerequisites:
        reach[a][b] = True
    for k in range(n):
        for i in range(n):
            if reach[i][k]:
                for j in range(n):
                    if reach[k][j]:
                        reach[i][j] = True
    return [sum(reach[u][v] for u in range(n)) for v in range(n)]

assert all_ancestors_ans(n1, p1) == [[], [0, 2, 3], [], [2]]
assert count_prereqs_ans(n1, p1) == [0, 3, 0, 1]
rng = random.Random(3)
for _ in range(200):
    n = rng.randint(2, 10)
    e = random_dag(n, 0.35, rng)
    anc = all_ancestors_ans(n, e)
    adj = build_adj(n, e)
    for v in range(n):
        assert anc[v] == sorted(u for u in range(n) if u != v and v in bfs_reach(adj, u))
    assert count_prereqs_ans(n, e) == [len(a) for a in anc]
print("Bài 3, 4: OK", all_ancestors_ans(n1, p1))

Bài 3, 4: OK [[], [0, 2, 3], [], [2]]


In [32]:
class Closure:
    def __init__(self, n: int):
        self.n = n
        self.anc = [1 << v for v in range(n)]   # tổ tiên (gồm chính nó)
        self.des = [1 << v for v in range(n)]   # hậu duệ (gồm chính nó)

    def add(self, a: int, b: int) -> None:
        if self.anc[b] >> a & 1:
            return                               # đã tới được, không đổi
        A, D = self.anc[a], self.des[b]
        for y in range(self.n):
            if D >> y & 1:
                self.anc[y] |= A
        for x in range(self.n):
            if A >> x & 1:
                self.des[x] |= D

    def query(self, u: int, v: int) -> bool:
        return u != v and bool(self.anc[v] >> u & 1)

rng = random.Random(11)
for _ in range(100):
    n = rng.randint(2, 10)
    edges = random_dag(n, 0.4, rng)
    rng.shuffle(edges)
    cl = Closure(n)
    added = []
    for a, b in edges:
        cl.add(a, b)
        added.append([a, b])
        u, v = rng.sample(range(n), 2)
        assert cl.query(u, v) == check_floyd(n, added, [[u, v]])[0]
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template Floyd–Warshall boolean (n nhỏ):**

```python
reach = [[False] * n for _ in range(n)]
for a, b in edges: reach[a][b] = True
for k in range(n):                 # k NGOÀI CÙNG
    for i in range(n):
        if reach[i][k]:
            for j in range(n):
                if reach[k][j]: reach[i][j] = True
```

**Template Kahn + bitmask tổ tiên:**

```python
pre = [0] * n
q = deque(i for i in range(n) if indeg[i] == 0)
while q:
    u = q.popleft()
    for v in adj[u]:
        pre[v] |= pre[u] | (1 << u)
        indeg[v] -= 1
        if indeg[v] == 0: q.append(v)
# u là tiên quyết của v  <=>  (pre[v] >> u) & 1
```

**Checklist:**
- [ ] Cạnh `[a, b]` là `a → b` (học `a` trước).
- [ ] Có tính **bắc cầu**, không chỉ cạnh trực tiếp.
- [ ] Floyd: vòng `k` ở ngoài cùng.
- [ ] Kahn: OR cả `pre[u]` lẫn `1 << u`.
- [ ] Kiểm tra bit bằng `>> u & 1`; trả về `bool`, đúng thứ tự truy vấn.
- [ ] Ma trận tạo bằng list comprehension, không `[[False]*n]*n`.